# 决策树规则提取器

本 notebook 演示两类决策树规则提取工具的使用流程：

| 工具 | 说明 |
|------|------|
| **DecisionTreeAnalyzer** | 标准 sklearn 决策树训练与评估（AUC/KS/LIFT 指标、save/load、规则提取） |
| **ManualTreeExtractor** | 人工干预决策树节点分裂（业务经验注入模型） |

**使用场景**：
- DecisionTreeAnalyzer：快速建立基线决策树，评估模型效果，提取可运营规则
- ManualTreeExtractor：数据驱动分裂不符合业务经验时，用人工阈值替换自动分裂

In [ ]:
import os, sys
sys.path.append('../')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hscredit.report.mining import DecisionTreeAnalyzer, ManualTreeExtractor
from hscredit import init_setting

init_setting()

## 1. 加载数据

In [ ]:
# 加载示例数据
df = pd.read_excel('hscredit_yyp.xlsx')

target = 'FPD'
print(f'样本数: {len(df):,}')
print(f'坏账率: {df[target].mean():.2%}')

# 选取用于建模的数值特征（排除ID、时间、标签列）
exclude_cols = ['客户编号', '放款时间', '商品类别', 'MOB1', 'CURRENT_DPD', target]
feature_list = [c for c in df.columns if c not in exclude_cols 
                and pd.api.types.is_numeric_dtype(df[c])]
print(f'建模特征 ({len(feature_list)} 个)')

# 划分训练集和测试集
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.3, random_state=42, stratify=df[target])
print(f'训练集: {len(df_train):,} 条, 坏账率: {df_train[target].mean():.2%}')
print(f'测试集: {len(df_test):,} 条, 坏账率: {df_test[target].mean():.2%}')

---

# Part A — DecisionTreeAnalyzer

**标准 sklearn 决策树包装器**，支持 AUC/KS/LIFT 等模型评估指标计算、规则提取与模型持久化。

## A1. 训练决策树

In [ ]:
# 初始化并训练决策树
fitter = DecisionTreeAnalyzer(
    target=target,
    feature_list=feature_list,
)
fitter.fit(df_train)

print(fitter)
print(f'叶子节点 ID: {fitter.get_leaf_node_ids()}')

## A2. 决策树可视化

训练完成后，使用 `hscredit.core.viz.plot_tree` 绘制决策树结构图，使用 `feature_importance_plot` 绘制特征重要性图：

## A3. 模型评估

`evaluate()` 支持 **AUC / KS / LIFT / TOP** 四种指标，同时计算训练集和多个测试集。


In [ ]:
from hscredit.core.viz import plot_tree, feature_importance_plot

# 绘制决策树结构图（matplotlib 后端）
output_dir = './model_report'
os.makedirs(output_dir, exist_ok=True)

fig_tree = plot_tree(
    fitter.clf,
    backend='matplotlib',
    feature_names=feature_list,
    class_names=['好', '坏'],
    save=f'{output_dir}/tree_analyzer_structure.png',
)
plt.show()

# 绘制特征重要性图
fig_imp = feature_importance_plot(
    features=feature_list,
    importance=fitter.clf.feature_importances_,
    top_n=12,
    figsize=(10, 6),
    save=f'{output_dir}/tree_analyzer_importance.png',
)
plt.show()

In [ ]:
# 多指标评估
for metric_type in ['auc', 'ks', 'lift', 'top']:
    result = fitter.evaluate(
        test_data_list=[('测试集', df_test)],
        metric_type=metric_type,
        top_rate=0.1  # lift/top 指标取 top 10%
    )
    for name, value in result:
        print(f'  {metric_type.upper():6s} | {name}: {value:.4f}')

## A4. 规则效果表

`get_rule_table()` 返回树中每个节点的统计指标：


In [ ]:
rule_table = fitter.get_rule_table()
rule_table.style.format({
    '样本占比': '{:.2%}',
    '坏账率': '{:.2%}',
    'LIFT值': '{:.4f}'
})

## A5. 按节点评估数据

`report()` 在新数据集上评估指定节点的命中效果：


In [ ]:
node_eval = fitter.report(df_test)
node_eval.style.format({
    '样本占比': '{:.2%}',
    '坏账率': '{:.2%}',
    'LIFT值': '{:.4f}'
})

## A6. 导出 Rule 对象并生成报告

`get_rules()` 将叶子节点规则转换为 `Rule` 对象，可进一步调用 `Rule.report()` 生成详细报告：


In [ ]:
rules = fitter.get_rules()
print(f'叶子规则共 {len(rules)} 条:\n')
for r in rules:
    print(f'[{r.name}]')
    print(f'  表达式: {r.expr}')
    print(f'  描述: {r.description}')
    print()

In [ ]:
# 找 LIFT 最高的叶子规则
leaf_rules = rule_table[rule_table['是否叶子'] == '是'].sort_values('LIFT值', ascending=False)
best = leaf_rules.iloc[0]
print(f'最高LIFT叶子: 节点 {int(best["节点编号"])}')
print(f'规则: {best["规则表达式"]}')
print(f'样本占比: {best["样本占比"]:.2%}, 坏账率: {best["坏账率"]:.2%}, LIFT: {best["LIFT值"]:.4f}')

# 找到对应 Rule 对象，生成详细报告
best_rule = next((r for r in rules if r.name == f'DecisionTree_N{int(best["节点编号"])}'), None)
if best_rule:
    report = best_rule.report(df_test, target=target)
    print('\n【Rule.report 详细报告】')
    report

## A7. 模型持久化

`save()` / `load()` 支持模型的保存和加载：


In [ ]:
fitter.save('/tmp/dt_model.pkl')
fitter_loaded = DecisionTreeAnalyzer.load('/tmp/dt_model.pkl')
print(f'加载后: {fitter_loaded}')
loaded_metrics = fitter_loaded.evaluate([('测试集', df_test)], metric_type='ks')
print(f'加载后 KS: {[(n, round(v,4)) for n,v in loaded_metrics]}')

---

# Part B — ManualTreeExtractor

**人工决策树提取器**，支持对 sklearn 决策树进行**人工指定分裂阈值**，
从而将业务经验注入数据驱动的模型。

## B1. 训练基础决策树 + 规则验证

训练基础决策树，绘制决策树结构图、特征重要性图，并在 df_train 上通过 `rule.report()` 独立验证每条规则的有效性。

In [ ]:
# 初始化并训练基础树
ext = ManualTreeExtractor(
    target=target,
    max_depth=2,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)
ext.fit(df_train, feature_list=feature_list)

# display() 一行展示：决策树 graphviz 图 + 美化规则表
ext.display()

print(ext)
print(f'总体样本数: {ext._n_total_samples:,}')
print(f'总体坏账率: {ext._overall_badrate:.2%}')

In [ ]:
# 不依赖树结构，在 df_train 上用 rule.report() 验证每条规则
print('【rule.report() 验证 — 在 df_train 上独立评估每条规则】\n')
rules_ext = ext.get_rules()
for rule in rules_ext:
    print(f"[{rule.name}] {rule.description}")
    display(rule.report(df_train, target=target))
    print()

## B2. 人工指定阈值分裂 + 规则验证

用业务经验阈值替换根分裂，并在 df_train 上通过 `rule.report()` 验证分裂后每条规则的有效性。

- `threshold=float` → 以指定阈值分裂
- `threshold=None` → 用决策树自动计算最优阈值
- `node` → 分裂的目标节点 ID（默认 0=根节点）

In [ ]:
# 人工指定阈值分裂
FEATURE = '身份证近一个月非银多头机构数'
THRESHOLD = 15

print(f'【人工指定阈值分裂】特征={FEATURE}, 阈值={THRESHOLD}, 节点=0（根节点）')
ext.manual_split(
    df=df_train,
    feature_name=FEATURE,
    threshold=THRESHOLD,
    node=0
)

# display() 一行展示：决策树 graphviz 图 + 美化规则表
ext.display()

In [ ]:
# 不依赖树结构，在 df_train 上用 rule.report() 验证每条规则
print('【rule.report() 验证 — 在 df_train 上独立评估每条规则】\n')
for rule in ext.get_rules():
    print(f"[{rule.name}] {rule.description}")
    display(rule.report(df_train, target=target))
    print()

## B3. 自动最优阈值分裂 + 规则验证

`threshold=None` 时自动计算最优分裂点，并在 df_train 上通过 `rule.report()` 验证每条规则的有效性。

In [ ]:
# 自动最优阈值分裂（threshold=None 时用决策树自动计算最优阈值）
ext_auto = ManualTreeExtractor(target=target, max_depth=3, random_state=42)
ext_auto.fit(df_train, feature_list=feature_list)

# display() 一行展示：决策树 graphviz 图 + 美化规则表
ext_auto.display()

print('【自动最优阈值分裂】根节点按 衡枢鉴真分老客版 自动寻找最优阈值')
ext_auto.manual_split(
    df=df_train,
    feature_name='衡枢鉴真分老客版',
    threshold=None,
    node=0
)
# display() 一行展示：决策树 graphviz 图 + 美化规则表
ext_auto.display()

In [ ]:
# 不依赖树结构，在 df_train 上用 rule.report() 验证每条规则
print('【rule.report() 验证 — 在 df_train 上独立评估每条规则】\n')
for rule in ext_auto.get_rules():
    print(f"[{rule.name}] {rule.description}")
    display(rule.report(df_train, target=target))
    print()

## B4. 链式调用：多层分裂 + 规则验证

`manual_split` 支持**链式调用**，可对不同节点依次进行分裂，并在 df_train 上通过 `rule.report()` 验证每条规则的有效性。

In [ ]:
# 链式分裂：多层逐步注入业务经验
ext_chain = ManualTreeExtractor(target=target, max_depth=4, random_state=42)
ext_chain.fit(df_train, feature_list=feature_list)
# display() 一行展示：决策树 graphviz 图 + 美化规则表
ext_chain.display()

# 第1步：根节点按身份证近一个月非银多头机构数 > 15 分裂
print('【链式分裂 — 第1步】')
ext_chain.manual_split(
    df=df_train[df_train['身份证近一个月非银多头机构数'] > 15],
    feature_name='身份证近一个月非银多头机构数',
    threshold=20,
    node=0
)
# display() 一行展示：决策树 graphviz 图 + 美化规则表
ext_chain.display()

# 第2步：在节点1（右侧子节点）继续按衡枢鉴真分分裂
print('【链式分裂 — 第2步】')
ext_chain.manual_split(
    df=df_train[df_train['身份证近一个月非银多头机构数'] > 20],
    feature_name='衡枢鉴真分老客版',
    threshold=None,
    node=1
)
# display() 一行展示：决策树 graphviz 图 + 美化规则表
ext_chain.display()

In [ ]:
# 不依赖树结构，在 df_train 上用 rule.report() 验证每条规则
print('【rule.report() 验证 — 在 df_train 上独立评估每条规则】\n')
for rule in ext_chain.get_rules():
    print(f"[{rule.name}] {rule.description}")
    display(rule.report(df_train, target=target))
    print()

## B5. 测试集评估 + 规则验证\n\n在测试集上评估链式分裂的规则效果，并在 df_test 上通过 `rule.report()` 独立验证。

In [ ]:
# 在测试集上评估链式分裂规则效果
eval_result = ext_chain.evaluate_on_new_data(df_test)
print(f'测试集样本: {len(df_test):,}, 总体坏账率: {df_test[target].mean():.2%}')
leaf_eval = eval_result[eval_result['是否叶子'] == '是']
leaf_eval.style.format({
    '样本占比': '{:.2%}',
    '坏账率': '{:.2%}',
    'LIFT值': '{:.4f}'
})

In [ ]:
# rule.report() 在测试集上独立验证
print('【rule.report() 验证 — 在 df_test 上独立评估每条规则】\n')
for rule in ext_chain.get_rules():
    print(f"[{rule.name}] {rule.description}")
    display(rule.report(df_test, target=target))
    print()

## B6. 规则效果对比

In [ ]:
# 数据驱动树 vs 人工调整树 — 叶子节点效果对比
auto_leaf = rule_table[rule_table['是否叶子'] == '是'].copy()
manual_leaf = leaf_eval.copy()

print('=== 数据驱动树（DecisionTreeAnalyzer）=== Leaf Nodes ===')
print(f'叶子节点数: {len(auto_leaf)}')
print(f'最高LIFT: {auto_leaf["LIFT值"].max():.4f}')
print(f'坏账率范围: {auto_leaf["坏账率"].min():.2%} ~ {auto_leaf["坏账率"].max():.2%}')

print('\n=== 人工调整树（ManualTreeExtractor）=== Leaf Nodes ===')
print(f'叶子节点数: {len(manual_leaf)}')
print(f'最高LIFT: {manual_leaf["LIFT值"].max():.4f}')
print(f'坏账率范围: {manual_leaf["坏账率"].min():.2%} ~ {manual_leaf["坏账率"].max():.2%}')

# 绘制对比图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, eval_df, title in zip(
    axes,
    [auto_leaf, manual_leaf],
    ['数据驱动树（DecisionTreeAnalyzer）', '人工调整树（ManualTreeExtractor）']
):
    eval_df = eval_df.sort_values('坏账率', ascending=False).reset_index(drop=True)
    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(eval_df)))
    bars = ax.bar(range(len(eval_df)), eval_df['坏账率'], color=colors)
    ax.axhline(df_test[target].mean(), color='gray', linestyle='--',
               label=f'总体坏账率 {df_test[target].mean():.2%}')
    ax.set_xticks(range(len(eval_df)))
    ax.set_xticklabels([f'N{int(row["节点编号"])}' for _, row in eval_df.iterrows()], rotation=45)
    ax.set_xlabel('节点编号')
    ax.set_ylabel('坏账率')
    ax.set_title(title)
    ax.legend()
    for bar, lift in zip(bars, eval_df['LIFT值']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'LIFT:{lift:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## 小结

| 工具 | 主要方法 | 说明 |
|------|----------|------|
| **DecisionTreeAnalyzer** | `fit()` | 训练标准 sklearn 决策树 |
|  | `evaluate(metric_type)` | AUC/KS/LIFT 评估 |
|  | `get_rule_table()` | 节点统计表 |
|  | `report()` | 按节点评估数据 |
|  | `get_rules()` | 叶子规则转 Rule 对象 |
|  | `save() / load()` | 模型持久化 |
| **ManualTreeExtractor** | `fit()` | 训练基础决策树 |
|  | `manual_split()` | 人工/自动分裂，支持链式调用 |
|  | `delete_node()` | 删除子树变叶子 |
|  | `get_rule_table()` | 节点统计表 |
|  | `evaluate_on_new_data()` | 在新数据上评估 |
|  | `get_rules()` | 叶子规则转 Rule 对象 |
|  | `display()` | 在 Jupyter 中展示决策树图 + 美化规则表 |